# Experiment 1: Performance, McNemar, and Stuart-Maxwell


In [1]:
import os
import pandas as pd
import json
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy.stats as st
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar, Table
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import plotly.graph_objects as go


import itertools

In [2]:
# ESAP questions with images
q_w_images = [3, 4, 6, 32, 35, 54, 68, 88]

cases = np.delete(np.arange(100), [0] + q_w_images) # valid questions
n_cases = cases.size

# Available options
options = ['A', 'B', 'C', 'D', 'E']
# even for the experiments without letter, manual annotator saved the option using letter.

path_to_results = "./../results"

model_file_name_dict = {'huatuo-o1': 'HuatuoGPT-o1',
                        'diabetica-o1': 'Diabetica-o1',
                        'diabetica-7B': 'Diabetica-7B',
                        'meditron3-8B': 'Meditron3-8B'}

In [3]:
def evaluate_response_distrib(filepath):
    """ 
    Given the excel file, this function parses and standardize the file entries.
    Returns a counter in the form of a dictionary.
    """
    df_ = pd.read_excel(filepath, header=None, index_col=0)
    
    # everything capital
    df_ = df_.map(lambda x: x.upper() if isinstance(x, str) else x) 
    
    # remove NaN values - this remove extra-columns
    df_ = df_.dropna(axis=1, how='all') 
    if not np.array_equal(df_.index, cases): # we keep all 91 cases
        raise ValueError("We are removing a valid case by mistake! Check Excel file")

    # unique annotations
    # print(df_.values)
    print(np.unique(df_.values))
    unique_elements = np.unique(df_.values) 

    # multiple options selected
    # are indicated as option1-option2: there is a dash
    bm_multiple = ["-" in v_ for v_ in unique_elements] 
    
    bm_hall_none = [True if (v_ == "HALL" or v_ == 'NONE') # hallucination or no response
                    else False for v_ in unique_elements]
    
    bm_others = ~np.logical_or(bm_multiple, bm_hall_none) # correct answers / check (temporary)
    
    idx_multiple = np.where(bm_multiple)[0] # indexes
    idx_hall_none = np.where(bm_hall_none)[0]
        
    dct_counter = {}
    all_ = 0 # to make sure we did not miss any response (91 or multiple for 10 rep)

    # we are creating a new label for multi and hall/none
    if len(idx_multiple) > 0:
        count = 0
        for v_ in unique_elements[idx_multiple]:
            count += (df_.values == v_).sum()
        all_ = count
        dct_counter["MULTI"] = count
            
    if len(idx_hall_none) > 0:
        count = 0
        for v_ in unique_elements[idx_hall_none]:
            count += (df_.values == v_).sum()
        all_ += count
        dct_counter["HALL / NONE"] = count
       
    # we are using the old label for everything else
    for element in unique_elements[bm_others]:
        count = (df_.values == element).sum()
        dct_counter[element] = count
        all_ += count
            
    list_add = ["CHECK", "MULTI", "HALL / NONE", "A", "B", "C", "D", "E"]
    for el_ in list_add:
        if el_ not in dct_counter.keys():
            dct_counter[el_] = 0
    dct_counter["TOTAL"] = all_
        
    return dct_counter

# Experiment 1: Check Performance & Results

Filenames for LLMs' outputs under prompt 1 and prompt 2.

In [4]:
model_paths_token = {'meditron3-8B-prompt1': 'meditron3-8B_promptID_001_output_4.xlsx',
                     'huatuo-o1-prompt1': 'huatuo-o1_promptID_001_output_9.xlsx',
                     'huatuo-o1-prompt2': 'huatuo-o1_promptID_002_output_11.xlsx',
                     'diabetica-o1-prompt1': 'diabetica-o1_promptID_001_output_2.xlsx',
                     'diabetica-o1-prompt2': 'diabetica-o1_promptID_002_output_3.xlsx',
                     'diabetica-7B-prompt1': 'diabetica-7B_promptID_001_output_3.xlsx',
                     'diabetica-7B-prompt2': 'diabetica-7B_promptID_002_output_4.xlsx',
                     'medfound7B-prompt1': 'medfound7B_promptID_001_output_2.xlsx',
                     'medfound7B-prompt2': 'medfound7B_promptID_002_output_3.xlsx',
                     'clinical-chatgpt-prompt1': 'clinical-chatgpt_promptID_001_output_2.xlsx',
                     'clinical-chatgpt-prompt2': 'clinical-chatgpt_promptID_002_output_3.xlsx',
                     'bloom-7b-prompt1': 'bloom-7B_promptID_001_output_0.xlsx',
                     'bloom-7b-prompt2': 'bloom-7B_promptID_002_output_1.xlsx',
                     'llama31-prompt1': 'llama31_promptID_001_output_6.xlsx',
                     'llama31-prompt2': 'llama31_promptID_002_output_7.xlsx',
                     'qwen2-7b-prompt1': 'qwen2-7b_promptID_001_output_0.xlsx',
                     'qwen2-7b-prompt2': 'qwen2-7b_promptID_002_output_1.xlsx',
                     'huatuo-o1-lms-prompt1': 'huatuo-o1-lms-prompt001.xlsx',
                     'gpt-oss-20b-prompt1': 'gpt-oss-20b-lms-prompt001.xlsx'
                    }

model_paths_no_token = {'huatuo-o1': 'huatuo-o1_promptID_001_output_47.xlsx',
                        'diabetica-o1': 'diabetica-o1_promptID_001_output_40.xlsx',
                        'diabetica-7B': 'diabetica-7B_promptID_001_output_47.xlsx',
                        'meditron3-8B': 'meditron3-8B_promptID_001_output_39.xlsx',
                        'clinical-chatgpt': 'clinical-chatgpt_promptID_001_output_4.xlsx',
                        'medfound7B': 'medfound7B_promptID_001_output_4.xlsx',
                        'bloom-7b': 'bloom-7b_promptID_001_output_2.xlsx',
                        'llama31': 'llama31_promptID_001_output_8.xlsx',
                        'qwen2-7b': 'qwen2-7b_promptID_001_output_2.xlsx'
                       }

### **Function to read excel and count entries**

In [5]:
def counter_extensive_dfs(result_folder, dct_models, letter=True):
    """ 
    Used in experiment 1 to a) keep track of distribution of responses
    b) build dataframe with extensive results.
    
    Variables: 
    result_folder: str, folder containing model's output
    dct_models: str, excel file 
    letter: bool, if True, ESAP question, otherwise, letter removed from option
    
    Returns:
    df_counter: pd.DataFrame, distribution of models responses
    df_all: pd.DataFrame extensive (91 outputs)
    """
    list_df = []
    index = []
    counting_outputs = []

    for model_, file_ in dct_models.items():
        if letter:
            filepath = join(path_to_results, model_.split("-prompt")[0], file_)
        else: 
            filepath = join(path_to_results, "NO_LETTERS", file_)
            
        df_ = pd.read_excel(filepath, header=None, index_col=0)
        df_ = df_[1] # we keep first column, others are empty
        list_df.append(df_)
        
        dct_counter = evaluate_response_distrib(filepath)
        counting_outputs.append([dct_counter[k] for k in sorted(dct_counter.keys())])

    # this is for the counter
    columns_name = sorted(dct_counter.keys())
    
    list_model_name = [m_ if letter else f"{m_}_no_letter" for m_ in dct_models.keys()]

    df_counter = pd.DataFrame(data=counting_outputs, 
                              index=list_model_name,
                              columns=columns_name)

    # this is for the Df with 91 responses
    df_all = pd.concat(list_df, axis=1)
    df_all.columns = list_model_name
    
    return df_counter, df_all

In [6]:
df_counter_prompt_A_B, df_all = counter_extensive_dfs(path_to_results, model_paths_token)

['A' 'B' 'C' 'D' 'E' 'NONE']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E' 'HALL']
['A' 'B' 'C' 'D' 'E']
['A' 'A-B' 'A-C' 'B' 'C' 'D' 'E' 'HALL' 'NONE']
['A' 'B' 'C' 'D' 'HALL' 'NONE']
['A' 'A-B-C-D-E' 'B' 'C' 'D' 'D-E' 'E' 'HALL' 'NONE']
['A' 'A-B' 'A-B-C-D-E' 'A-C' 'B' 'C' 'D' 'E' 'NONE']
['A' 'A-B' 'A-C' 'B' 'C' 'D' 'HALL' 'NONE']
['A' 'A-B' 'A-B-C' 'A-B-C-D' 'A-B-C-D-E' 'A-D' 'B' 'C' 'D' 'HALL' 'NONE']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'C-D' 'D' 'E']
['A' 'B' 'C' 'D' 'E']
['A' 'B' 'C' 'D' 'E' 'NONE']


In [7]:
df_counter_prompt_A_B

,A,B,C,CHECK,D,E,HALL / NONE,MULTI,TOTAL
meditron3-8B-prompt1,18,21,18,0,19,13,2,0,91
huatuo-o1-prompt1,19,24,23,0,13,12,0,0,91
huatuo-o1-prompt2,21,19,22,0,17,12,0,0,91
diabetica-o1-prompt1,24,24,20,0,17,6,0,0,91
diabetica-o1-prompt2,15,23,21,0,20,12,0,0,91
diabetica-7B-prompt1,13,19,28,0,19,11,1,0,91
diabetica-7B-prompt2,13,21,26,0,19,12,0,0,91
medfound7B-prompt1,13,1,1,0,1,1,72,2,91
medfound7B-prompt2,30,3,1,0,1,0,56,0,91
clinical-chatgpt-prompt1,23,19,17,0,7,6,16,3,91


In [8]:
df_all.head()

,meditron3-8B-prompt1,huatuo-o1-prompt1,huatuo-o1-prompt2,diabetica-o1-prompt1,diabetica-o1-prompt2,diabetica-7B-prompt1,diabetica-7B-prompt2,medfound7B-prompt1,medfound7B-prompt2,clinical-chatgpt-prompt1,clinical-chatgpt-prompt2,bloom-7b-prompt1,bloom-7b-prompt2,llama31-prompt1,llama31-prompt2,qwen2-7b-prompt1,qwen2-7b-prompt2,huatuo-o1-lms-prompt1,gpt-oss-20b-prompt1
0,,,,,,,,,,,,,,,,,,,
1,B,B,B,B,B,HALL,D,HALL,HALL,B,C,NONE,HALL,B,B,D,C,B,B
2,A,C,A,A,E,D,D,HALL,NONE,D,D,B,A,D,D,A,D,C,E
5,A,C,A,B,C,C,B,NONE,NONE,A,A,D,A,B,B,B,C,C,A
7,C,E,E,C,C,D,A,NONE,NONE,D,A,A,A,C,C,C,C,D,A
8,A,A,A,A,C,C,C,NONE,NONE,A,A,A,A,A,D,A,C,B,C


In [9]:
df_all.shape, df_counter_prompt_A_B.shape

((91, 19), (19, 9))

In [10]:
model_paths_no_token

{'huatuo-o1': 'huatuo-o1_promptID_001_output_47.xlsx',
 'diabetica-o1': 'diabetica-o1_promptID_001_output_40.xlsx',
 'diabetica-7B': 'diabetica-7B_promptID_001_output_47.xlsx',
 'meditron3-8B': 'meditron3-8B_promptID_001_output_39.xlsx',
 'clinical-chatgpt': 'clinical-chatgpt_promptID_001_output_4.xlsx',
 'medfound7B': 'medfound7B_promptID_001_output_4.xlsx',
 'bloom-7b': 'bloom-7b_promptID_001_output_2.xlsx',
 'llama31': 'llama31_promptID_001_output_8.xlsx',
 'qwen2-7b': 'qwen2-7b_promptID_001_output_2.xlsx'}

In [11]:
df_counter_no_let, df_all_no_let = counter_extensive_dfs(path_to_results, model_paths_no_token, letter=False)

['A' 'B' 'C' 'C-D' 'D' 'E' 'NONE']
['A' 'A-B' 'B' 'C' 'C-D' 'D' 'E' 'HALL' 'NONE']
['A' 'A-B' 'B' 'B-C' 'C' 'C-D' 'C-E' 'D' 'E' 'HALL' 'NONE']
['A' 'A-B' 'B' 'B-D' 'C' 'D' 'E' 'HALL' 'NONE']
['A' 'A-E' 'B' 'C' 'D' 'E' 'HALL' 'NONE']
['A' 'A-B-C' 'A-B-C-D-E' 'A-E' 'B' 'C' 'D' 'E' 'HALL' 'NONE']
['A' 'A-B' 'B' 'B-C' 'C' 'D' 'E' 'E-NONE' 'HALL' 'NONE']
['A' 'B' 'C' 'D' 'E' 'NONE']
['A' 'A-B' 'B' 'C' 'D' 'E']


In [12]:
os.makedirs("tables", exist_ok=True)
pd.concat([df_counter_prompt_A_B, df_counter_no_let]).to_latex("tables/readings_experiment_1_20-04-2026.tex")

In [13]:
# concatenate all results from experiment 1
df_all = pd.concat([df_all, df_all_no_let], axis=1)

Loading ground-truth responses from ESAP, after being manually converted into json file.

In [14]:
with open("./../inference/Ped-ESAP.json", "r") as f:
    content = json.load(f)
    
responses = []
labs = []
table = []
for k, v in content.items():
    if len(v['answer']) > 0:
        responses.append(v['answer'][0])
        labs.append(v["labs"]=='Yes')
        table.append(v["table"]=='Yes')
    else:
        responses.append(None)
        labs.append(None)
        table.append(None)
        
ground_truth = pd.DataFrame(data=np.array(responses[:-1]).reshape(-1,1), index=np.arange(1,100),
                           columns=["truth"])
ground_truth = ground_truth.drop(q_w_images)

meta_labs_table = np.hstack((np.array(labs).reshape(-1,1), np.array(table).reshape(-1,1)))
meta_labs_tab_df = pd.DataFrame(data=meta_labs_table[:-1, :], index=np.arange(1,100),
                    columns=["labs", "table"])
meta_labs_tab_df = meta_labs_tab_df.drop(q_w_images)

In [15]:
meta_labs_tab_df[meta_labs_tab_df["table"]]

,labs,table
9,True,True
24,True,True
36,True,True
73,False,True


Ground truth is concatenated to table as last column.

In [16]:
# run only once, otherwise we get multiple "truth" columns
df_all = pd.concat([df_all, ground_truth], axis=1)

In [17]:
# here we keep the value in entry if is one among A, B, C, D, or E
# we report as NaN otherwise
for col in df_all.columns[:-1]:
    df_all[col] = df_all[col].where(df_all[col].isin(options), np.nan)

In [18]:
# df_all.to_csv("./tables/output_llms.csv")

In [19]:
df_all.head()

,meditron3-8B-prompt1,huatuo-o1-prompt1,huatuo-o1-prompt2,diabetica-o1-prompt1,diabetica-o1-prompt2,diabetica-7B-prompt1,diabetica-7B-prompt2,medfound7B-prompt1,medfound7B-prompt2,clinical-chatgpt-prompt1,...,huatuo-o1_no_letter,diabetica-o1_no_letter,diabetica-7B_no_letter,meditron3-8B_no_letter,clinical-chatgpt_no_letter,medfound7B_no_letter,bloom-7b_no_letter,llama31_no_letter,qwen2-7b_no_letter,truth
1,B,B,B,B,B,NaN,D,NaN,NaN,B,...,B,C,D,B,D,NaN,NaN,C,D,B
2,A,C,A,A,E,D,D,NaN,NaN,D,...,C,A,D,NaN,D,NaN,A,B,D,A
5,A,C,A,B,C,C,B,NaN,NaN,A,...,B,B,B,B,A,NaN,A,B,B,C
7,C,E,E,C,C,D,A,NaN,NaN,D,...,A,E,A,E,D,NaN,NaN,E,C,A
8,A,A,A,A,C,C,C,NaN,NaN,A,...,A,B,NaN,A,A,NaN,NaN,A,A,E


In [20]:
bm_valid = ~df_all.isna()

In [21]:
accuracy_df = df_all[df_all.columns[:-1]].eq(df_all['truth'], axis=0)

accuracy_df.head()

,meditron3-8B-prompt1,huatuo-o1-prompt1,huatuo-o1-prompt2,diabetica-o1-prompt1,diabetica-o1-prompt2,diabetica-7B-prompt1,diabetica-7B-prompt2,medfound7B-prompt1,medfound7B-prompt2,clinical-chatgpt-prompt1,...,gpt-oss-20b-prompt1,huatuo-o1_no_letter,diabetica-o1_no_letter,diabetica-7B_no_letter,meditron3-8B_no_letter,clinical-chatgpt_no_letter,medfound7B_no_letter,bloom-7b_no_letter,llama31_no_letter,qwen2-7b_no_letter
1,True,True,True,True,True,False,False,False,False,True,...,True,True,False,False,True,False,False,False,False,False
2,True,False,True,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
5,False,True,False,False,True,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7,False,False,False,False,False,False,True,False,False,False,...,True,True,False,True,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


We sum each column to obtain the accuracy score.

In [22]:
accuracy_scores = accuracy_df.sum()

In [24]:
models_col = df_all.columns[:-1] # we exclude the truth column

We call interpretable the case where a model is able to give a unique answer, which is one of the options specified by ESAP.

#### Double checking - the number in the following table corresponds with the more extensive ones in the article (Appendix)

In [25]:
results = pd.concat([accuracy_scores[models_col], bm_valid[models_col].sum()], axis=1)
results.columns = ['# correct', '# interpretable']

In [26]:
results/n_cases

,# correct,# interpretable
meditron3-8B-prompt1,0.329670,0.978022
huatuo-o1-prompt1,0.351648,1.000000
huatuo-o1-prompt2,0.351648,1.000000
diabetica-o1-prompt1,0.329670,1.000000
diabetica-o1-prompt2,0.340659,1.000000
diabetica-7B-prompt1,0.296703,0.989011
diabetica-7B-prompt2,0.318681,1.000000
medfound7B-prompt1,0.043956,0.186813
medfound7B-prompt2,0.120879,0.384615
clinical-chatgpt-prompt1,0.197802,0.791209


In [27]:
groups = {}
for m_ in model_paths_no_token.keys():
    groups[f"{m_}_prompt"] = [f"{m_}-prompt1", f"{m_}-prompt2"]
    groups[f"{m_}_token"] = [f"{m_}-prompt1", f"{m_}_no_letter"]
groups

{'huatuo-o1_prompt': ['huatuo-o1-prompt1', 'huatuo-o1-prompt2'],
 'huatuo-o1_token': ['huatuo-o1-prompt1', 'huatuo-o1_no_letter'],
 'diabetica-o1_prompt': ['diabetica-o1-prompt1', 'diabetica-o1-prompt2'],
 'diabetica-o1_token': ['diabetica-o1-prompt1', 'diabetica-o1_no_letter'],
 'diabetica-7B_prompt': ['diabetica-7B-prompt1', 'diabetica-7B-prompt2'],
 'diabetica-7B_token': ['diabetica-7B-prompt1', 'diabetica-7B_no_letter'],
 'meditron3-8B_prompt': ['meditron3-8B-prompt1', 'meditron3-8B-prompt2'],
 'meditron3-8B_token': ['meditron3-8B-prompt1', 'meditron3-8B_no_letter'],
 'clinical-chatgpt_prompt': ['clinical-chatgpt-prompt1',
  'clinical-chatgpt-prompt2'],
 'clinical-chatgpt_token': ['clinical-chatgpt-prompt1',
  'clinical-chatgpt_no_letter'],
 'medfound7B_prompt': ['medfound7B-prompt1', 'medfound7B-prompt2'],
 'medfound7B_token': ['medfound7B-prompt1', 'medfound7B_no_letter'],
 'bloom-7b_prompt': ['bloom-7b-prompt1', 'bloom-7b-prompt2'],
 'bloom-7b_token': ['bloom-7b-prompt1', 'bloom

In [28]:
def bootstrap_kappa(x, y, B=10000, n=n_cases):
    # bootstrap for Cohen's coefficient confidence intervals
    rng = np.random.default_rng(seed=np.random.choice(100000))
    idx = np.arange(n)
    kappas = []
    for b in range(B): 
        I = rng.choice(idx, len(idx), True)
        tmp_agreement_mask = (x == y) & x.notna() & y.notna()
        tmp_n_agree = tmp_agreement_mask.iloc[I].sum()
        kappas.append((tmp_n_agree/n-0.2)/0.8)
    lo, hi = np.percentile(kappas, [2.5, 97.5])
    return lo, hi


accuracy_when_agree = {}

    
for model, cols in groups.items():
    if len(cols) == 2:
        col1, col2 = cols
        if not (col1 in df_all.columns and col2 in df_all.columns):
            continue
            
        # Keep rows where both predictions agree and are not NaN
        agreement_mask = (df_all[col1] == df_all[col2]) & df_all[col1].notna() & df_all[col2].notna()
        agreed_preds = df_all.loc[agreement_mask, col1]
        truths = df_all.loc[agreement_mask, 'truth']
        x, y = df_all[col1], df_all[col2]
        low, high = bootstrap_kappa(x, y)
        # Calculate accuracy where predictions agree
        acc = (agreed_preds == truths).sum()
        accuracy_when_agree[model] = {
            "n_agreements": agreement_mask.sum(),
            "# match & correct": acc,
            "bootstrap CI": (low, high)
        }

# Display results
for model, stats in accuracy_when_agree.items():
    lo, hi = proportion_confint(stats['n_agreements'], n_cases, method="wilson")
    print(f"""{model.upper()}\n
    {stats['n_agreements']} agreements, 
    # match & correct = {stats['# match & correct']:.3f}, 
    match-rate = {stats['n_agreements']/n_cases:.2f}, 95% CI [{lo:.2f}, {hi:.2f}],
    Cohen's coef = {(stats['n_agreements']/n_cases-0.2)/0.8},
    Bootstrap CI Cohen's = {stats['bootstrap CI']}\n\n""")

HUATUO-O1_PROMPT

    58 agreements, 
    # match & correct = 23.000, 
    match-rate = 0.64, 95% CI [0.53, 0.73],
    Cohen's coef = 0.5467032967032966,
    Bootstrap CI Cohen's = (np.float64(0.423076923076923), np.float64(0.6703296703296704))


HUATUO-O1_TOKEN

    44 agreements, 
    # match & correct = 19.000, 
    match-rate = 0.48, 95% CI [0.38, 0.58],
    Cohen's coef = 0.35439560439560436,
    Bootstrap CI Cohen's = (np.float64(0.23076923076923078), np.float64(0.47802197802197804))


DIABETICA-O1_PROMPT

    47 agreements, 
    # match & correct = 17.000, 
    match-rate = 0.52, 95% CI [0.42, 0.62],
    Cohen's coef = 0.39560439560439564,
    Bootstrap CI Cohen's = (np.float64(0.27197802197802196), np.float64(0.5192307692307693))


DIABETICA-O1_TOKEN

    44 agreements, 
    # match & correct = 16.000, 
    match-rate = 0.48, 95% CI [0.38, 0.58],
    Cohen's coef = 0.35439560439560436,
    Bootstrap CI Cohen's = (np.float64(0.23076923076923078), np.float64(0.47802197802197804))

In [30]:
def count_ci(k, n=n_cases):
    lo, hi = proportion_confint(k, n, method="wilson")
    return pd.Series({
        "acc": k / n,
        "lo_prop": lo,
        "hi_prop": hi,
        "lo_count": lo * n,
        "hi_count": hi * n
    })

new = results["# correct"].copy()
# Apply across the DataFrame
new = new.apply(count_ci)

# Combine back with your model labels
df_ci = pd.concat([results, new], axis=1)
df_ci

,# correct,# interpretable,acc,lo_prop,hi_prop,lo_count,hi_count
meditron3-8B-prompt1,30,89,0.329670,0.241709,0.431430,21.995519,39.260105
huatuo-o1-prompt1,32,91,0.351648,0.261373,0.453942,23.784923,41.308685
huatuo-o1-prompt2,32,91,0.351648,0.261373,0.453942,23.784923,41.308685
diabetica-o1-prompt1,30,91,0.329670,0.241709,0.431430,21.995519,39.260105
diabetica-o1-prompt2,31,91,0.340659,0.251514,0.442713,22.887746,40.286870
diabetica-7B-prompt1,27,90,0.296703,0.212635,0.397240,19.349776,36.148872
diabetica-7B-prompt2,29,91,0.318681,0.231960,0.420091,21.108360,38.228272
medfound7B-prompt1,4,17,0.043956,0.017224,0.107631,1.567417,9.794415
medfound7B-prompt2,11,35,0.120879,0.068855,0.203615,6.265813,18.528964
clinical-chatgpt-prompt1,18,72,0.197802,0.128945,0.291140,11.733959,26.493762


In the following, we show the case index where we had agreement across prompt A, prompt B, and prompt A without letter token.

In [31]:
for m_ in model_paths_no_token.keys():
    acc_pmt1 = np.where(accuracy_df[f'{m_}-prompt1'])[0]
    acc_no_let = np.where(accuracy_df[f'{m_}_no_letter'])[0]
    if m_ != 'meditron3-8B':
        acc_pmt2 = np.where(accuracy_df[f'{m_}-prompt2'])[0]
        print(m_)
        print(sorted(list((set(acc_pmt2).intersection(set(acc_pmt1))).intersection(set(acc_no_let)))))
        print("\n")

huatuo-o1
[np.int64(0), np.int64(6), np.int64(8), np.int64(21), np.int64(24), np.int64(38), np.int64(40), np.int64(54), np.int64(55), np.int64(56), np.int64(65), np.int64(68), np.int64(69), np.int64(71), np.int64(75), np.int64(83), np.int64(90)]


diabetica-o1
[np.int64(6), np.int64(20), np.int64(24), np.int64(40), np.int64(50), np.int64(59), np.int64(62), np.int64(68), np.int64(69), np.int64(77), np.int64(89), np.int64(90)]


diabetica-7B
[np.int64(6), np.int64(15), np.int64(19), np.int64(22), np.int64(34), np.int64(35), np.int64(40), np.int64(55), np.int64(59), np.int64(62), np.int64(68), np.int64(69), np.int64(83), np.int64(85), np.int64(89)]


clinical-chatgpt
[np.int64(5), np.int64(23), np.int64(40), np.int64(43), np.int64(71)]


medfound7B
[np.int64(71)]


bloom-7b
[]


llama31
[np.int64(6), np.int64(9), np.int64(14), np.int64(21), np.int64(22), np.int64(24), np.int64(35), np.int64(38), np.int64(40), np.int64(41), np.int64(54), np.int64(55), np.int64(61), np.int64(62), np.int64(6

In [32]:
models = [["huatuo-o1-prompt1", "huatuo-o1-prompt2", "huatuo-o1_no_letter"],
          ["diabetica-o1-prompt1", "diabetica-o1-prompt2", "diabetica-o1_no_letter"], 
          ["diabetica-7B-prompt1", "diabetica-7B-prompt2", "diabetica-7B_no_letter"], 
          ["meditron3-8B-prompt1", "meditron3-8B_no_letter"],
          ["clinical-chatgpt-prompt1", "clinical-chatgpt-prompt2", "clinical-chatgpt_no_letter"],
          ["medfound7B-prompt1", "medfound7B-prompt2", "medfound7B_no_letter"],
          ["bloom-7b-prompt1", "bloom-7b-prompt2", "bloom-7b_no_letter"],
          ["llama31-prompt1", "llama31-prompt2", "llama31_no_letter"],
          ["qwen2-7b-prompt1", "qwen2-7b-prompt2", "qwen2-7b_no_letter"]
          ]

## McNemar's Test

In [33]:
# McNemar's test
def mcnemar_single(series1_bm, series2_bm): 
    x, y = series1_bm.to_numpy(), series2_bm.to_numpy()
    n10 = np.sum((x==1)&(y==0))
    n01 = np.sum((x==0)&(y==1))
    table = [[np.sum((x==0)&(y==0)), n01],
             [n10, np.sum((x==1)&(y==1))]]
    pvals = mcnemar(table, exact=True).pvalue
    return pvals

    
for cols in models: 
    pairs = list(itertools.combinations(cols, 2))
    for a, b in pairs:
        p = mcnemar_single(accuracy_df[a], accuracy_df[b])
        print(f"{a} vs {b}: p={p:.4f}")
    print("\n")

huatuo-o1-prompt1 vs huatuo-o1-prompt2: p=1.0000
huatuo-o1-prompt1 vs huatuo-o1_no_letter: p=0.8388
huatuo-o1-prompt2 vs huatuo-o1_no_letter: p=0.8318


diabetica-o1-prompt1 vs diabetica-o1-prompt2: p=1.0000
diabetica-o1-prompt1 vs diabetica-o1_no_letter: p=0.4049
diabetica-o1-prompt2 vs diabetica-o1_no_letter: p=0.3075


diabetica-7B-prompt1 vs diabetica-7B-prompt2: p=0.7539
diabetica-7B-prompt1 vs diabetica-7B_no_letter: p=0.8238
diabetica-7B-prompt2 vs diabetica-7B_no_letter: p=0.5034


meditron3-8B-prompt1 vs meditron3-8B_no_letter: p=1.0000


clinical-chatgpt-prompt1 vs clinical-chatgpt-prompt2: p=1.0000
clinical-chatgpt-prompt1 vs clinical-chatgpt_no_letter: p=1.0000
clinical-chatgpt-prompt2 vs clinical-chatgpt_no_letter: p=1.0000


medfound7B-prompt1 vs medfound7B-prompt2: p=0.0923
medfound7B-prompt1 vs medfound7B_no_letter: p=1.0000
medfound7B-prompt2 vs medfound7B_no_letter: p=0.0654


bloom-7b-prompt1 vs bloom-7b-prompt2: p=1.0000
bloom-7b-prompt1 vs bloom-7b_no_letter: p=0.0

## Stuart-Maxwell Test

In [34]:
def stuart_maxwell(x, y): 
    tab = pd.crosstab(x, y)
    res = Table(tab).test_nominal_association()  # Stuart–Maxwell
    return res.statistic, res.pvalue
    
for cols in models: 
    print(cols[0].split('-prompt')[0])
    pairs = list(itertools.combinations(cols, 2))
    pvals = []
    for a,b in pairs:
        stats, pval = stuart_maxwell(df_all[a], df_all[b])
        print(f"{a} vs {b}: p={pval}")
    print("\n")

huatuo-o1
huatuo-o1-prompt1 vs huatuo-o1-prompt2: p=9.992007221626409e-16
huatuo-o1-prompt1 vs huatuo-o1_no_letter: p=2.645000279777321e-05
huatuo-o1-prompt2 vs huatuo-o1_no_letter: p=1.170857699683836e-08


diabetica-o1
diabetica-o1-prompt1 vs diabetica-o1-prompt2: p=2.421497805160655e-06
diabetica-o1-prompt1 vs diabetica-o1_no_letter: p=1.949451960669979e-05
diabetica-o1-prompt2 vs diabetica-o1_no_letter: p=4.433931888314646e-09


diabetica-7B
diabetica-7B-prompt1 vs diabetica-7B-prompt2: p=0.0
diabetica-7B-prompt1 vs diabetica-7B_no_letter: p=3.2980825848127893e-09
diabetica-7B-prompt2 vs diabetica-7B_no_letter: p=1.8207657603852567e-13


meditron3-8B
meditron3-8B-prompt1 vs meditron3-8B_no_letter: p=0.005977985631178462


clinical-chatgpt
clinical-chatgpt-prompt1 vs clinical-chatgpt-prompt2: p=0.1898673855785693
clinical-chatgpt-prompt1 vs clinical-chatgpt_no_letter: p=0.004637283163608008
clinical-chatgpt-prompt2 vs clinical-chatgpt_no_letter: p=0.342828844242973


medfound7B
medf